In [ ]:
import pandas as pd
from datasets import Dataset, ClassLabel
import numpy as np
import random
import torch
import os
from transformers import AutoTokenizer
from transformers import AutoModelForSequenceClassification
from peft import LoraConfig, get_peft_model
from transformers import TrainingArguments
from transformers import Trainer
from transformers import EarlyStoppingCallback
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.metrics import confusion_matrix, classification_report

In [ ]:
OUTPUT_MODEL_NAME = "synth_lora_model_distilbert"
CHECKPOINT_DIR = "checkpoints_distilbert"
TENSORBOARD_RUN_NAME = "synthetic_data_experiment_distilbert"

In [ ]:
# Set random states.
def set_random_states(random_state):
    # Set various random seeds.
    np.random.seed(random_state)
    random.seed(random_state)
    torch.manual_seed(random_state)
    torch.cuda.manual_seed_all(random_state)
    os.environ["PYTHONHASHSEED"] = str(random_state)
    os.environ["TOKENIZERS_PARALLELISM"] = "false"
    try:
        torch.use_deterministic_algorithms(True)
    except Exception:
        pass
    return random_state
RANDOM_STATE = set_random_states(1618)

# Make constant variables.
MODEL_NAME = "distilbert-base-uncased" # do 'microsoft/MiniLM-L12-H384-uncased' next

In [ ]:
# Load the dataset
df = pd.read_csv("./dataSyntheticAll.csv")
dataset = Dataset.from_pandas(df)

In [ ]:
# Map dataset labels to 1s or 0s.
label_map = {
    "met": 0,
    "unmet": 1
}

dataset = dataset.map(lambda x: {"label": label_map[x["needs"]]})


label_feature = ClassLabel(names=["met", "unmet"])
dataset = dataset.cast_column("label", label_feature)

In [ ]:
# Tokenize texts.
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(example):
    return tokenizer(
        example["report"],
        truncation=True,
        padding="max_length",
        max_length=256
    )

dataset = dataset.map(tokenize)
 # Set PyTorch format. 
dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "label"]
)

In [ ]:
# Split the dataset. (80/10/10)
dataset = dataset.train_test_split(test_size=0.2, seed=RANDOM_STATE, stratify_by_column="label")

train_dataset = dataset["train"]
temp_dataset = dataset["test"]

temp_split = temp_dataset.train_test_split(test_size=0.5, seed=RANDOM_STATE, stratify_by_column="label")

val_dataset = temp_split["train"]
test_dataset = temp_split["test"]

# Check distributions.
def check_distribution(dataset, name):
    df = dataset.to_pandas()
    counts = df["label"].value_counts(normalize=True)
    print(f"{name} distribution:")
    print(counts)

check_distribution(train_dataset, "Train")
check_distribution(val_dataset, "Validation")
check_distribution(test_dataset, "Test")


In [ ]:
# Confirm device for use. 
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
# Load the base model.
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

# Define LoRA config.
lora_config = LoraConfig(
    r=8, # LoRA attention dimension (rank)
    lora_alpha=16, # alpha for LoRA scaling
    target_modules=["q_lin", "v_lin"], # specific named modules to be replaced
    lora_dropout=0.05, # dropout probability for LoRA layers
    bias="none", # bias type
    task_type="SEQ_CLS" # what type of task (sequence classification)
)

# Attach LoRA to model. 
model = get_peft_model(model, lora_config)
model.to(device)
model.print_trainable_parameters()

In [ ]:
# Set training arguments.
training_args = TrainingArguments(
    output_dir=CHECKPOINT_DIR,
    learning_rate=2e-5,
    num_train_epochs=10, # higher so early stopping can trigger
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="tensorboard",
    run_name=TENSORBOARD_RUN_NAME,
    fp16=True
)

# Add early stopping.
early_stopping = EarlyStoppingCallback(
    early_stopping_patience=2
)

# Add metrics function.
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        preds,
        average="binary"
    )
    acc = accuracy_score(labels, preds)
    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }

# Create trainer.
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    callbacks = [early_stopping],
    compute_metrics=compute_metrics
)

In [ ]:
# Train the model.
trainer.train()

In [ ]:
# Save the fine-tuned model.
model.save_pretrained(OUTPUT_MODEL_NAME)
tokenizer.save_pretrained(OUTPUT_MODEL_NAME)

In [ ]:
# Move model to eval mode.
model.eval()

# Get predictions.
preds, labels = [], []

for batch in test_dataset:
    inputs = {k: torch.tensor([v]) for k, v in batch.items() if k in ["input_ids", "attention_mask"]}
    
    with torch.no_grad():
        outputs = model(**inputs)
    
    pred = torch.argmax(outputs.logits, dim=1).item()
    preds.append(pred)
    labels.append(batch["label"])

preds = np.array(preds)
labels = np.array(labels)

# Confusion Matrix
cm = confusion_matrix(labels, preds)
print(" --------------- Confusion Matrix --------------- ")
print(cm)

# Classification Report
report = classification_report(labels, preds, target_names=model.config.id2label.values())
print(" --------------- Classification Report --------------- ")
print(report)